In [1]:
#!pip install imblearn
#!pip install optuna
#!pip install ydata_profiling

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import seaborn as sns
import numpy as np
from imblearn.over_sampling import SMOTE
from collections import Counter
import copy
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import cross_val_score, KFold, cross_validate
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt

%matplotlib inline

In [3]:
cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/4a73ca4928f2b95f960cd9b9f44c4700244ed553/data/raw/patient_train_data.csv', 
                        encoding='UTF-8',
                        index_col=0,
                        sep=',',
                        on_bad_lines='skip', 
                        quoting=3)
cancer_df.head(3)

,Alcohol Consumption,Cancer Stage,Country,Date of Birth,Diabetes,Diabetes History,Diet Risk,Early Detection,Family History,Gender,...,Non Smoker,Obesity BMI,Physical Activity,Screening History,Smoking History,Transfusion History,Treatment Type,Tumor Size (mm),Urban or Rural,Survival Prediction
ID,,,,,,,,,,,,,,,,,,,,,
1,No,Localized,UK,29-01-1966,No,No,Moderate,No,No,M,...,Yes,Overweight,Low,Regular,No,-,Chemotherapy,33.0,Urban,Yes
2,No,Regional,Japan,21-12-1958,No,No,Low,No,No,M,...,No,Normal,Low,Irregular,Yes,-,Chemotherapy,17.0,Urban,No
3,No,Localized,France,16-06-1959,No,No,Low,Yes,No,M,...,No,Normal,Moderate,Never,Yes,-,Surgery,34.0,Urban,Yes


In [4]:
for column in ['Diabetes', 'Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease']:
    print(cancer_df.loc[:,column].value_counts(normalize=True))

Diabetes
No     0.800816
Yes    0.199184
Name: proportion, dtype: float64
Diabetes History
No     0.999867
Yes    0.000133
Name: proportion, dtype: float64
Heart Disease History
No     0.898381
Yes    0.101619
Name: proportion, dtype: float64
Inflammatory Bowel Disease
No     0.899979
Yes    0.100021
Name: proportion, dtype: float64


# PREDICTING IN TEST SET

In [5]:
#!pip install import_ipynb

In [6]:
import import_ipynb
import prep_functions as prep

In [7]:
columns_to_delete = ['Transfusion History', 'Marital Status', 'Smoking History', 'Diabetes History']

values_to_imput_cat = {
        'Healthcare Access': '?',
        'Gender': 'P'
    }

binary_cols = [
        'Heart Disease History', 'Inflammatory Bowel Disease',
        'Survival Prediction', 'Diabetes', 'Alcohol Consumption', 'Early Detection',
        'Family History', 'Genetic Mutation'
    ]

numeric_cols = [
        'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K',
        'Tumor Size (mm)'
    ]

categorical_cols = [
        'Cancer Stage', 'Country', 'Diet Risk', 'Gender', 'Healthcare Access',
        'Insurance Costs', 'Insurance Status', 'Obesity BMI', 'Physical Activity',
        'Screening History', 'Non Smoker', 'Treatment Type', 'Urban or Rural'
]

#columns_to_balance = ['Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease']

mode_train = {}

stats_pre = {}

In [8]:
test_cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/refs/heads/master/data/raw/patient_test_data.csv', 
                        encoding='UTF-8',
                        index_col=0,
                        sep=',',
                        on_bad_lines='skip', 
                        quoting=3)

test_cancer_df.head(1)

,Alcohol Consumption,Cancer Stage,Country,Date of Birth,Diabetes,Diabetes History,Diet Risk,Early Detection,Family History,Gender,...,Mortality Rate per 100K,Non Smoker,Obesity BMI,Physical Activity,Screening History,Smoking History,Transfusion History,Treatment Type,Tumor Size (mm),Urban or Rural
ID,,,,,,,,,,,,,,,,,,,,,
75036,Yes,Localized,UK,17-11-1947,No,No,Low,Yes,No,M,...,5.0,Yes,Overweight,Low,Regular,No,-,Combination,69.0,Urban


In [9]:
X_train, X_test, y_train, y_test = prep.split_sets(cancer_df)

Dimension of X_train: (60028, 30)
Dimension of X_test: (15007, 30)
Dimension of y_train: (60028,)
Dimension of y_test: (15007,)


In [10]:
# Following line includes columns_to_balance as an argument (temporaryly commented within the function)
#X_train_processed, y_train_processed, mode_train, stats_pre = prep.preprocess_train_df(X_train, y_train, columns_to_delete, values_to_imput_cat, binary_cols, numeric_cols, categorical_cols, columns_to_balance)

X_train_processed, y_train_processed, mode_train, stats_pre = prep.preprocess_train_df(X_train, y_train, columns_to_delete, values_to_imput_cat, binary_cols, numeric_cols, categorical_cols)

KeyError: 'Heart_Disease_History'

In [ ]:
X_test_processed = prep.preprocess_test_df(test_cancer_df, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols)

In [ ]:
X_test_processed = X_test_processed[X_train_processed.columns]

In [ ]:
df_kaggle.head()

,Survival Prediction
0,Yes
1,Yes
2,Yes
3,Yes
4,Yes


In [ ]:
df_kaggle = pd.DataFrame(y_pred, index=test_cancer_df.index)

df_kaggle.replace({0: 'No', 1: 'Yes'}, inplace = True)

df_kaggle.columns = ['Survival Prediction']

df_kaggle.head()

df_kaggle.to_csv('DT_Group05_Version03.csv')